# Projeto para Turismo - Planejador de Viagens (multiagente)

## Instalação das bibliotecas

In [1]:
!pip install crewai crewai[tools] crewai[google-genai] python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Carregamento do modelo

In [41]:
from crewai import LLM
import os

id_model = "gemini/gemini-3.8-flash" # @param {type:"string"}

gemini = LLM(
    model = id_model,
    api_key=os.getenv("GOOGLE_API_KEY")
)

## Criação das Tools (ferramentas)

In [5]:
!pip install -q duckduckgo-search ddgs langchain_community


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [6]:
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults

/tmp/ipykernel_30694/3914554391.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


In [7]:
@tool
def search_web_tools(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = DuckDuckGoSearchResults(num_results=4, verbose=True)
    return search_tool.run(query)

In [8]:
from langchain_community.tools import DuckDuckGoSearchRun
search = DuckDuckGoSearchRun()
search.invoke("eventos em londres no próximo mês")

'Grandes Ofertas Em Shows E Musicais Do West End De Londres. Melhores Preços Online. Aproveite As Melhores Ofertas De Última Hora Para Shows E Musicais No West End Londres. Tipos de Espectáculos, Peças de Teatro, Concertos Teatrais, Peças de teatro and more Faça suas reservas, receba confirmação e baixe seus ingressos no celular de forma prática. Londres Espetáculos e musicais: Melhores de 2026 - Cancelamento Grátis Reserve Ingressos Para Shows E Musicais Imperdíveis No West End De Londres. Garanta Seus Bilhetes Agora E Aproveite Experiências Inesquecíveis Em Londres Líder Mundial · Pagamento 100% Seguro · Guia em Português · Reserve Agora e Sem Filas Encontre eventos a acontecer a no próximo mês em Londres, Reino Unido. Explore uma variedade de atividades e interesses para planear o seu dia fora de casa perfeito. La guía más completa de qué hacer este mes en Londres: 48+ eventos — conciertos, fiestas, gastronomía, planes gratis y al aire libre. Actualizado octubre 2026.'

### Criação de classe

In [27]:
!pip install langchain-tavily


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [10]:
from langchain_tavily import TavilySearch
from langchain_community.tools import DuckDuckGoSearchResults

class SearchTools:
    @tool("Pesquisa na internet")
    def search_tavily(query: str = "") -> str:
        """
        Search the web using TavilyAPI.
        Recommended for more structured and recent information.
        """
        search_tavily = TavilySearch(max_results = 4)
        search_res = search_tavily.invoke(input=query)
        return search_res

    @tool("Pesquisa na internet com DuckDuckGo")
    def search_duckduckgo(query: str):
        """
        Search the web using DuckDuckGo.
        Returns a list of search results.
        """
        search_tool = DuckDuckGoSearchResults(num_results=4, verbose=True)
        return search_tool.run(query)        

In [11]:
SearchTools.search_tavily

Tool(name='Pesquisa na internet', description='\n        Search the web using TavilyAPI.\n        Recommended for more structured and recent information.\n        ', env_vars=[], args_schema=<class 'crewai.tools.base_tool.Pesquisanainternet'>, result_schema=None, description_updated=False, cache_function=<function _default_cache_function at 0x7fc093436fc0>, result_as_answer=False, max_usage_count=None, tool_failure_policy=None, current_usage_count=0, func=<function SearchTools.search_tavily at 0x7fc05a14d580>, tool_type='crewai.tools.base_tool.Tool')

### Outras ferramentas

In [12]:
class CalculatorTools:
    @tool("Faça um cálculo")
    def calculate(operation):
        """
        Useful to perform any mathematical calculations,
        like sum, minus, multiplication, division, etc.
        The input to this tool should be a mathematical expression, a 
        couple examples are `200*7` or `5000/2*10`
        """
        try:
            return eval(operation)
        except SyntaxError:
            return "Erro: Sintaxe inválida"

CalculatorTools.calculate

Tool(name='Faça um cálculo', description='\n        Useful to perform any mathematical calculations,\n        like sum, minus, multiplication, division, etc.\n        The input to this tool should be a mathematical expression, a \n        couple examples are `200*7` or `5000/2*10`\n        ', env_vars=[], args_schema=<class 'crewai.tools.base_tool.Façaumcálculo'>, result_schema=None, description_updated=False, cache_function=<function _default_cache_function at 0x7fc093436fc0>, result_as_answer=False, max_usage_count=None, tool_failure_policy=None, current_usage_count=0, func=<function CalculatorTools.calculate at 0x7fc06a7000e0>, tool_type='crewai.tools.base_tool.Tool')

## Criação de Agentes

In [13]:
from textwrap import dedent

texto = """
    Este é um texto
    com identação errada.
"""
print(texto)


    Este é um texto
    com identação errada.



### Especialista em informações da cidade

In [43]:
from crewai import Agent

city_info_agent = Agent(
    role = "Especialista em informações da cidade",
    goal = dedent(
        """
            Reunir informações úteis sobre o destino escolhido, com base nos interesses e datas de viagem, ajudando viajantes a entender o contexto geral do local.
        """
    ),
    backstory = dedent(
        """
            Sou um especialista com amplo conhecimento sobre cidades ao redor do mundo,
            capaz de fornecer detalhes atualizados sobre clima, eventos culturais, segurança,
            costumes locais e outros aspectos práticos para quem pretende visitar o local. 
            Sou apaixonado por compartilhar as melhores experiências e 'joias escondidas' do local.
        """
    ),
    llm = gemini,
    tools = [SearchTools.search_tavily],
    verbose = True,
    max_iter = 10,
    allow_delegation = False
)

### Logística e informações da cidade

In [44]:
logistics_expert_agent = Agent(
    role = "Especialista em logística de viagem",
    goal = dedent(
        """
            Identificar as melhores opções logísticas para a viagem, com foco em praticidade, custo-benefício e conforto.
        """
    ),
    backstory = dedent(
        """
            Sou um profissional focado em planejar a logística da viagem: transporte, hospedagem e deslocamento local.
            Tenho conhecimento sobre companhias aéreas, apps de mobilidade, regiões seguras para se hospedar e otimização de trajetos.
        """
    ),
    llm=gemini,
    tools=[
        SearchTools.search_tavily,
        CalculatorTools.calculate
    ],
    verbose=True,
    max_iter=5,
    allow_delegation=False
)

### Especialista em criação de itinerário

In [45]:
itinerary_planner_agent = Agent(
    role="Planejador de itinerário personalizado",
    goal="Criar um roteiro completo com base nas preferências do usuário.",
    backstory=dedent(
        """
        Sou um guia profissional apaixonado por viagens, especialista em organização de itinerários personalizados.
        Minha missão é integrar dados sobre clima, atrações, eventos e logística em uma experiência otimizada para o turista.
        """
    ),
    llm=gemini,
    tools=[SearchTools.search_tavily],
    verbose=True,
    max_iter=5,
    allow_delegation=False
)

## Criação das tarefas (Tasks)

In [ ]:
from crewai import Task

from_city = "São Paulo" # @param {type: "string"}
destination_city = "Londres" # @param {type: "string"}
date_from = "23 de novembro de 2025" # @param {type: "string"}
date_to = "30 de novembro de 2025" # @param {type: "string"}
interests = "arte, tecnologia, vinhos, arquitetura" # @param {type: "string"}

print(f"""
Viajando de: {from_city}
Para: {destination_city}
Chegada: {date_from}
Partida: {date_to}
""")


Viajando de: São Paulo
Para: Londres
Chegada: 23 de novembro de 2025
Partida: 30 de novembro de 2025



### Task: Coleta de informações da cidade

In [19]:
def city_info_task(agent, from_city, destination_city, interests, date_from, date_to):
    return Task(
        description=dedent(
            f"""
            Levantar informações detalhadas sobre a cidade e coletar dados úteis sobre clima, segurança e costumes locais.
            Identificar marcos culturais, pontos históricos, locais de entretenimento, experiências gastronômicas e quaisquer atividades que se alinhem às preferências do usuário.
            Também destacar eventos e festivais sazonais que podem ser de interesse durante a visita do viajante.
            Use as ferramentas disponíveis para buscar fontes atualizadas e confiáveis.

            Viajando de: {from_city}
            Para: {destination_city}
            Interesses do viajante: {interests}
            Chegada: {date_from}
            Partida: {date_to}

            Seja criterioso, como se estivesse ajudando alguém a ter uma experiência inesquecível.
            """
        ),
        expected_output=dedent(
            f"""
            Um guia detalhado (em formato markdown) em português que inclui:
            - Resumo da cidade e sua cultura;
            - Uma lista selecionada de lugares recomendados para visitar e eventos (se houver);
            - Um detalhamento das despesas diárias, como custos médios com alimentação;
            - Recomendações de segurança;
            - Dicas de costumes locais;
            """
        ),
        agent=agent,
        output_file='relatorio_local.md'
    )

### Task: Planejamento de logística

In [20]:
def plan_logistics_task(context, agent, destination_city, interests, date_from, date_to):
    return Task(
        description=dedent(
            f"""
            Planejar a logística da viagem.
            Identificar as melhores opções de hospedagem, voos e transporte local considerando:

            Destino: {destination_city}
            Data de chegada: {date_from}
            Data de partida: {date_to}
            Interesses: {interests}

            Avalie preço, localização, conveniência e segurança.
            """
        ),
        expected_output=dedent(
            f"""
            Relatório em português (em formato markdown):
            - Sugestão de hospedagens, preferencialmente com localização estratégica;
            - Opções de voos ou meios de transporte para chegada/partida;
            - Sugestões de deslocamento dentro da cidade;
            - Estimativas de custo para cada item;
            """
        ),
        context=context,
        agent=agent,
        output_file='relatorio_logistica.md',
    )

### Task: Construção de itinerário final

In [21]:
def build_itinerary_task(context, agent, destination_city, interests, date_from, date_to):
    return Task(
        description=dedent(
            f"""
            Esta tarefa sintetiza todas as informações para criar o roteiro final da viagem.
            Com base nas informações dos outros agentes, desenvolva um itinerário detalhado.
            Cada dia deve conter atividades, clima, transporte, refeições e estimativa de gastos.

            Destino: {destination_city}
            Interesses: {interests}
            Data de chegada: {date_from}
            Data de partida: {date_to}
            """
        ),
        expected_output=dedent("""
        Documento em português que inclui (em formato markdown):
        - Boas vindas e apresentação da cidade em até 2 parágrafos.
        - Programação diária com sugestões de horários e atividades
        - Atrações distribuídas por região e logística
        - Clima previsto, sugestões de transporte e alimentação
        - Eventos e festivais sazonais (se houver)
        - Estimativa de gastos por dia, custo médio das despesas diárias e pontos turísticos.
        - Visão geral dos destaques da cidade com base nas recomendações do guia.
        - Outras dicas e informações adicionais para uma viagem e estadia tranquila.
        """),
        context=context,
        agent=agent,
        output_file='roteiro_viagem.md',
    )

### Executando a criação das tasks

In [48]:
city_info = city_info_task(
    city_info_agent,
    from_city,
    destination_city,
    interests,
    date_from,
    date_to
)

In [49]:
plan_logistics = plan_logistics_task(
    [city_info],
    logistics_expert_agent,
    destination_city,
    interests,
    date_from,
    date_to
)

In [50]:
build_itinerary = build_itinerary_task(
  [city_info, plan_logistics],
  itinerary_planner_agent,
  destination_city,
  interests,
  date_from,
  date_to,
)

## Criação da equipe (Crew)

In [51]:
from crewai import Crew, Process

crew = Crew(
  agents=[city_info_agent, logistics_expert_agent, itinerary_planner_agent],
  tasks=[city_info, plan_logistics, build_itinerary],
  process=Process.sequential,
  full_output=True,
  share_crew=False,
  verbose=True,
  max_rpm=10
)
result = await crew.kickoff_async()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 70074ad7-0a92-4fcb-8fe5-f61375bc20c9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Levantar informações detalhadas sobre a cidade e coletar dados úteis sobre clima, segurança e costumes         │
│  locais.                                                                                                        │
│  Identificar marcos culturais, pontos históricos, locais de entretenimento, experiências gastronômicas e        │
│  quaisquer atividades que se alinhem às preferências do usuário.                                                │
│  Também destacar eventos e festivais sazonais que podem ser de interesse durante a visita do viajante.          │
│  Use as ferramentas disponíveis para buscar fontes atualizadas e confiáveis.                                    │
│                                                                                                                 │
│  Viajando de: São Paulo                                                                                         │
│  Para: Londres                                                                                                  │
│  Interesses do viajante: arte, tecnologia, vinhos, arquitetura                                                  │
│  Chegada: 23 de novembro de 2025                                                                                │
│  Partida: 30 de novembro de 2025                                                                                │
│                                                                                                                 │
│  Seja criterioso, como se estivesse ajudando alguém a ter uma experiência inesquecível.                         │
│                                                                                                                 │
│  ID: 1a913dbc-cc95-4f98-a229-3829ff8b71e6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Especialista em informações da cidade                                                                   │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Levantar informações detalhadas sobre a cidade e coletar dados úteis sobre clima, segurança e costumes         │
│  locais.                                                                                                        │
│  Identificar marcos culturais, pontos históricos, locais de entretenimento, experiências gastronômicas e        │
│  quaisquer atividades que se alinhem às preferências do usuário.                                                │
│  Também destacar eventos e festivais sazonais que podem ser de interesse durante a visita do viajante.          │
│  Use as ferramentas disponíveis para buscar fontes atualizadas e confiáveis.                                    │
│                                                                                                                 │
│  Viajando de: São Paulo                                                                                         │
│  Para: Londres                                                                                                  │
│  Interesses do viajante: arte, tecnologia, vinhos, arquitetura                                                  │
│  Chegada: 23 de novembro de 2025                                                                                │
│  Partida: 30 de novembro de 2025                                                                                │
│                                                                                                                 │
│  Seja criterioso, como se estivesse ajudando alguém a ter uma experiência inesquecível.                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: pesquisa_na_internet                                                                                     │
│  Args: {'query': 'London events exhibitions late November 2025 art architecture'}                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool pesquisa_na_internet executed with result: {'query': 'London events exhibitions late November 2025 art architecture', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://artreview.com/the-10-exhibitions-to-s...


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Planejador de itinerário personalizado                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Esta tarefa sintetiza todas as informações para criar o roteiro final da viagem.                               │
│  Com base nas informações dos outros agentes, desenvolva um itinerário detalhado.                               │
│  Cada dia deve conter atividades, clima, transporte, refeições e estimativa de gastos.                          │
│                                                                                                                 │
│  Destino: Londres                                                                                               │
│  Interesses: arte, tecnologia, vinhos, arquitetura                                                              │
│  Data de chegada: 23 de novembro de 2025                                                                        │
│  Data de partida: 30 de novembro de 2025                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Planejador de itinerário personalizado                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Roteiro Exclusivo: Londres Cultural, Contemporânea & Sensorial                                               │
│  **Período:** 23 a 30 de Novembro de 2025 (7 noites / 8 dias)                                                   │
│  **Origem:** São Paulo (GRU) | **Destino:** Londres, Reino Unido (LHR)                                          │
│  **Foco:** Arte, Tecnologia, Vinhos e Arquitetura                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Boas-vindas e Apresentação de Londres                                                                    │
│                                                                                                                 │
│  Bem-vindo a Londres, uma das capitais culturais mais fascinantes e dinâmicas do planeta. Em nenhum outro       │
│  lugar do mundo o diálogo entre o passado imperial e a vanguarda contemporânea se manifesta com tanta           │
│  intensidade: arranha-céus de arquitetura *high-tech* projetam suas sombras sobre muralhas romanas e pontes     │
│  seculares, enquanto antigas usinas de energia e galpões industriais abrigam os centros de arte contemporânea   │
│  e tecnologia imersiva mais provocativos do nosso tempo. Em fins de novembro, a cidade ganha uma atmosfera      │
│  cinematográfica única: o frescor do outono tardio cede espaço às primeiras brisas de inverno, o entardecer     │
│  chega por volta das 16h trazendo o acendimento das famosas iluminações natalinas, e a cultura dos pubs e       │
│  acolhedores *wine bars* à luz de velas atinge seu apogeu.                                                      │
│                                                                                                                 │
│  Neste itinerário feito sob medida para apreciadores de arte, arquitetura, design, inovação e boa gastronomia   │
│  com foco em vinhos, cada deslocamento foi milimetricamente agrupado por eixos geográficos para eliminar        │
│  correrias e trajetos desnecessários. Prepare-se para vivenciar desde instalações digitais em telas 8K e        │
│  marcos do Brutalismo até coleções particulares secretas, exposições de gala de mestres da pintura e taças      │
│  memoráveis em caves históricas e bistrôs premiados. Londres está pronta para surpreender você em cada          │
│  esquina.                                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Visão Geral dos Destaques & Eixos Logísticos                                                             │
│                                                                                                                 │
│  Para maximizar seu tempo e garantir deslocamentos conf

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Esta tarefa sintetiza todas as informações para criar o roteiro final da viagem.                               │
│  Com base nas informações dos outros agentes, desenvolva um itinerário detalhado.                               │
│  Cada dia deve conter atividades, clima, transporte, refeições e estimativa de gastos.                          │
│                                                                                                                 │
│  Destino: Londres                                                                                               │
│  Interesses: arte, tecnologia, vinhos, arquitetura                                                              │
│  Data de chegada: 23 de novembro de 2025                                                                        │
│  Data de partida: 30 de novembro de 2025                                                                        │
│                                                                                                                 │
│  Agent: Planejador de itinerário personalizado                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 70074ad7-0a92-4fcb-8fe5-f61375bc20c9                                                                       │
│  Final Output: # Roteiro Exclusivo: Londres Cultural, Contemporânea & Sensorial                                 │
│  **Período:** 23 a 30 de Novembro de 2025 (7 noites / 8 dias)                                                   │
│  **Origem:** São Paulo (GRU) | **Destino:** Londres, Reino Unido (LHR)                                          │
│  **Foco:** Arte, Tecnologia, Vinhos e Arquitetura                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Boas-vindas e Apresentação de Londres                                                                    │
│                                                                                                                 │
│  Bem-vindo a Londres, uma das capitais culturais mais fascinantes e dinâmicas do planeta. Em nenhum outro       │
│  lugar do mundo o diálogo entre o passado imperial e a vanguarda contemporânea se manifesta com tanta           │
│  intensidade: arranha-céus de arquitetura *high-tech* projetam suas sombras sobre muralhas romanas e pontes     │
│  seculares, enquanto antigas usinas de energia e galpões industriais abrigam os centros de arte contemporânea   │
│  e tecnologia imersiva mais provocativos do nosso tempo. Em fins de novembro, a cidade ganha uma atmosfera      │
│  cinematográfica única: o frescor do outono tardio cede espaço às primeiras brisas de inverno, o entardecer     │
│  chega por volta das 16h trazendo o acendimento das famosas iluminações natalinas, e a cultura dos pubs e       │
│  acolhedores *wine bars* à luz de velas atinge seu apogeu.                                                      │
│                                                                                                                 │
│  Neste itinerário feito sob medida para apreciadores de arte, arquitetura, design, inovação e boa gastronomia   │
│  com foco em vinhos, cada deslocamento foi milimetricamente agrupado por eixos geográficos para eliminar        │
│  correrias e trajetos desnecessários. Prepare-se para vivenciar desde instalações digitais em telas 8K e        │
│  marcos do Brutalismo até coleções particulares secretas, exposições de gala de mestres da pintura e taças      │
│  memoráveis em caves históricas e bistrôs premiados. Londres está pronta para surpreender você em cada          │
│  esquina.                                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Visão Geral dos Destaques & Eixos Logísticos                                                             │
│                                                                                                                 │
│  Para maximizar seu tempo e garantir deslocamentos con

## Visulizar resultado

In [52]:
from IPython.display import display, Markdown

def load_markdown(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            content = content.replace("```markdown","").replace("```","")
            return content
    except Exception as e:
        print(f"Erro ao carregar o arquivo: {str(e)}")
        return None

In [53]:
def display_markdown(content):
    return display(Markdown(content))

In [54]:
md = load_markdown("./roteiro_viagem.md")
display_markdown(md)

# Roteiro Exclusivo: Londres Cultural, Contemporânea & Sensorial
**Período:** 23 a 30 de Novembro de 2025 (7 noites / 8 dias)  
**Origem:** São Paulo (GRU) | **Destino:** Londres, Reino Unido (LHR)  
**Foco:** Arte, Tecnologia, Vinhos e Arquitetura  

---

## 1. Boas-vindas e Apresentação de Londres

Bem-vindo a Londres, uma das capitais culturais mais fascinantes e dinâmicas do planeta. Em nenhum outro lugar do mundo o diálogo entre o passado imperial e a vanguarda contemporânea se manifesta com tanta intensidade: arranha-céus de arquitetura *high-tech* projetam suas sombras sobre muralhas romanas e pontes seculares, enquanto antigas usinas de energia e galpões industriais abrigam os centros de arte contemporânea e tecnologia imersiva mais provocativos do nosso tempo. Em fins de novembro, a cidade ganha uma atmosfera cinematográfica única: o frescor do outono tardio cede espaço às primeiras brisas de inverno, o entardecer chega por volta das 16h trazendo o acendimento das famosas iluminações natalinas, e a cultura dos pubs e acolhedores *wine bars* à luz de velas atinge seu apogeu.

Neste itinerário feito sob medida para apreciadores de arte, arquitetura, design, inovação e boa gastronomia com foco em vinhos, cada deslocamento foi milimetricamente agrupado por eixos geográficos para eliminar correrias e trajetos desnecessários. Prepare-se para vivenciar desde instalações digitais em telas 8K e marcos do Brutalismo até coleções particulares secretas, exposições de gala de mestres da pintura e taças memoráveis em caves históricas e bistrôs premiados. Londres está pronta para surpreender você em cada esquina.

---

## 2. Visão Geral dos Destaques & Eixos Logísticos

Para maximizar seu tempo e garantir deslocamentos confortáveis nos dias frios de novembro, o planejamento foi estruturado em quatro eixos temáticos e logísticos principais:

* **Eixo Sul (South Bank, Bankside & Bermondsey):** O coração da arte contemporânea e da gastronomia à beira-rio, abrigando a imponente **Tate Modern**, a icônica passarela **Millennium Bridge**, os queijos e vinhos do **Borough Market** e as caves de **Bermondsey**.
* **Eixo Leste & City (Arquitetura High-Tech & Brutalismo):** O contraste visceral entre as torres futuristas de Norman Foster e Richard Rogers (**The Gherkin**, **Lloyd's Building**, **The Garden at 120**) e a utopia modernista do **Barbican Estate**, finalizando na cena de vinhos naturais do leste londrino (**Sager + Wilde**).
* **Eixo Histórico & Coleções Exclusivas (Holborn, Westminster & Pimlico):** A genialidade arquitetônica do **Sir John Soane’s Museum**, o encontro épico de **Turner e Constable na Tate Britain** e a atmosfera centenária à luz de velas do **Gordon’s Wine Bar**.
* **Eixo Oeste & Central (Design, Imersão Tecnológica & Luzes de Inverno):** A cobertura escultural do **Design Museum** (com a mostra de Wes Anderson), as instalações digitais imersivas da **Frameless** e do **Outernet London**, a elegância da **Somerset House**, as iluminações sazonais de **Carnaby & Regent Street** e as cartas excepcionais do **Noble Rot**.

---

## 3. Clima Previsto, Vestuário e Festivais Sazonais

### Clima e Luz Natural em Fins de Novembro
* **Temperaturas:** Médias entre **5°C e 11°C**. Dias frios com ar úmido e possibilidade de chuviscos leves e passageiros (*drizzle* britânico).
* **Luz do Sol:** Nascer do sol por volta das **07h40** e pôr do sol cedo, por volta das **16h00**. A partir do meio da tarde, o acendimento das luzes e decorações transforma as ruas num espetáculo visual contínuo.
* **Vestuário Recomendado:** Estratégia de camadas (*layering*): camiseta/segunda pele térmica, malha de lã ou cashmere, e um casaco externo corta-vento/impermeável. Calçados confortáveis e à prova d'água são indispensáveis, acompanhados de cachecol e luvas leves para caminhadas ao ar livre.

### Eventos e Destaques Sazonais Especiais (Novembro 2025)
* **Luzes de Natal Urbanas:** Circuitos luminosos já completamente ativos em **Regent Street**, **Carnaby Street** (com instalações temáticas e criativas) e **Covent Garden**.
* **Pista de Patinação da Somerset House:** O neoclássico pátio central do século XVIII transformado em um rinque de patinação glamouroso com música e quiosques de *mulled wine* (vinho quente com especiarias).
* **Hyde Park Winter Wonderland:** O maior festival de inverno da Europa, combinando barraquinhas de artesanato, vilas alpinas e iluminação cênica.
* **Grandes Exposições Temporárias:** 
  * *Wes Anderson: The Archives* no **Design Museum**.
  * *Turner and Constable* na **Tate Britain**.

---

## 4. Programação Diária Detalhada

*Base de Hospedagem Sugerida: Região de Bankside / Southwark (ex: citizenM London Bankside ou The Hoxton Southwark).*

---

### Dia 1: Domingo, 23/11/2025 – Boas-vindas, Chegada e Primeiras Luzes ao Longo do Tâmisa
* **Região Principal:** Bankside / Southwark  
* **Clima Previsto:** 6°C a 10°C, céu encoberto, sensação de frio no entardecer.

* **14h00 – 16h00 | Chegada e Check-in Descomplicado:**  
  Desembarque em Londres (Heathrow), traslado direto de cerca de 35 a 45 minutos pela moderna **Elizabeth Line** até o centro e check-in no hotel na região de Bankside. Momento para acomodar as malas, tomar um banho revigorante e se vestir em camadas para a primeira saída.
* **16h30 – 18h30 | Caminhada Arquitetônica pelo South Bank ao Pôr do Sol:**  
  Saída a pé em direção ao calçadão do Rio Tâmisa. Observe a imponente estrutura de tijolos da antiga usina de Bankside contrastando com a silhueta metálica da **Millennium Bridge** (desenhada por Norman Foster com Anthony Caro) e a cúpula iluminada da **Catedral de St. Paul** refletida nas águas do rio.
* **19h00 – 21h30 | Jantar de Boas-Vindas & Carta de Vinhos:**  
  Jantar descontraído no restaurante e wine bar **Seabird** (no terraço do The Hoxton Southwark, com vista panorâmica da cidade e excelente carta de vinhos brancos e frutos do mar) ou no acolhedor bistrô do **citizenM**.
* **Transporte:** Elizabeth Line (traslado aeroporto) e deslocamento 100% a pé no período noturno.
* **Estimativa de Gastos:**
  * *Transporte:* £13,30 (Elizabeth Line de Heathrow).
  * *Alimentação:* £45 a £70 (jantar de boas-vindas com vinho).
  * *Atrações:* £0 (circuito a pé aberto).
  * *Total do Dia:* **~£58 a £83**.

---

### Dia 2: Segunda-feira, 24/11/2025 – O Templo da Arte Moderna, Aromas do Borough e o Horizonte da City
* **Região Principal:** Bankside, Southwark e City of London  
* **Clima Previsto:** 5°C a 9°C, sol entre nuvens, vento fresco junto ao rio.

* **08h30 – 09h45 | Café da Manhã de Especialidade:**  
  Café no *Origin Coffee* ou *Monmouth Coffee* nas proximidades de Bankside.
* **10h00 – 13h00 | Imersão na Tate Modern (Arte & Arquitetura Industrial):**  
  Visite o mais célebre museu de arte moderna da Grã-Bretanha. Comece pelo colossal **Turbine Hall** e explore as galerias da coleção permanente (Picasso, Rothko, Warhol, Louise Bourgeois). Suba até o terraço panorâmico do **Blavatnik Building** (projeto dos arquitetos suíços Herzog & de Meuron) para admirar a engenharia do complexo e uma vista de 360° da metrópole.
* **13h15 – 15h30 | Almoço Sensorial no Borough Market & Degustação no Bedales:**  
  Caminhe 7 minutos até o histórico **Borough Market**. Após explorar as bancas de trufas, pães artesanais e queijos britânicos, acomode-se no **Bedales of Borough**, um wine bar intimista encravado no mercado. Desfrute de uma tábua de queijos de pequenos produtores britânicos e charcutaria fina harmonizada com vinhos de mínima intervenção.
* **16h00 – 17h30 | Travessia da London Bridge e Mirante The Garden at 120:**  
  Cruze o Tâmisa rumo à *Square Mile* (a City). Visite o **The Garden at 120** (120 Fenchurch St) – um terraço ajardinado público, aberto e gratuito no 15º andar, que oferece um olhar cara a cara com o The Shard e os arranha-céus da City no momento exato em que as luzes do crepúsculo se acendem.
* **18h30 – 21h30 | Jantar & Brinde com Vinhos Europeus Clássicos:**  
  Jantar no charmoso bistrô **The French House** no Soho ou retorno para Southwark no conceituado pub gastronômico **The Anchor & Hope** (famoso pela gastronomia sazonal britânica e seleção refinada de tintos da Borgonha e do Vale do Rhône).
* **Transporte:** Caminhadas curtas a pé e 1 viagem de metrô (Jubilee / Northern Line).
* **Estimativa de Gastos:**
  * *Transporte:* £3,50 a £8,50 (teto diário TfL).
  * *Alimentação:* £60 a £90 (almoço degustação no Bedales + jantar).
  * *Atrações:* £0 (Tate Modern acervo permanente gratuito; Garden at 120 gratuito).
  * *Total do Dia:* **~£65 a £100**.

---

### Dia 3: Terça-feira, 25/11/2025 – Arquitetura Brutalista, Ousadia High-Tech e Vanguarda dos Vinhos no Leste
* **Região Principal:** Barbican, City of London e Shoreditch / Bethnal Green  
* **Clima Previsto:** 5°C a 10°C, tempo seco com aberturas de sol pela manhã.

* **09h00 – 11h30 | A Utopia de Concreto do Barbican Estate:**  
  Desembarque na estação Barbican. Explore o complexo residencial e centro de artes **Barbican Centre**, projetado pela firma Chamberlin, Powell and Bon nos anos 1960. Percorra as passarelas elevadas de concreto martelado, o lago central com espelhos d'água e a icônica estufa tropical (**Barbican Conservatory**). Uma masterclass de Brutalismo em escala monumental.
* **12h00 – 13h30 | Safári Arquitetônico na City: Lloyd’s & The Gherkin:**  
  Caminhe até o coração financeiro. Pare diante do revolucionário **Lloyd’s of London** (projeto de Richard Rogers com o conceito *inside-out*, onde tubulações, dutos de ar e elevadores panorâmicos de vidro ficam na fachada externa) e contemple a geometria helicoidal do **The Gherkin (30 St Mary Axe)** de Norman Foster.
* **13h30 – 15h00 | Almoço Histórico no Leadenhall Market:**  
  Almoce sob as abóbadas vitorianas de ferro fundido do **Leadenhall Market** (estrutura do século XIX pintada em verde e dourado). Parada no bistrô *Luc’s Brasserie* ou no wine bar *Old Tom’s Bar*.
* **15h30 – 18h00 | Design Urbano e Murais em Shoreditch:**  
  Siga em direção ao leste descolado (*East End*). Passeie pelas ruas repletas de intervenções artísticas de rua, galerias independentes e pop-ups conceituais ao redor de Redchurch Street.
* **18h30 – 21h30 | Templo dos Vinhos Naturais: Sager + Wilde:**  
  Dirija-se ao cultuado **Sager + Wilde** (em Hackney / Bethnal Green Road). Este é o bar que redefiniu o consumo de vinho em Londres: ambiente despojado com balcão feito de antigas lajotas de ferro fundido de metrô, servindo safras biodinâmicas raras em taça, vinhos naturais do Jura e da Califórnia, acompanhados de clássicos sanduíches artesanais de queijo curado (*cheese toasties* de alta gastronomia) e tábuas curadas.
* **Transporte:** Metrô (District/Circle Line e Elizabeth Line ou Northern Line) dentro da Zona 1-2.
* **Estimativa de Gastos:**
  * *Transporte:* £8,50 (Daily Cap).
  * *Alimentação:* £60 a £95 (almoço em Leadenhall + degustação e jantar no Sager + Wilde).
  * *Atrações:* £0 (Barbican Estate e tour exterior de arquitetura gratuitos).
  * *Total do Dia:* **~£70 a £105**.

---

### Dia 4: Quarta-feira, 26/11/2025 – Ilusionismo Arquitetônico, Mestres da Luz e Caves Históricas de 1890
* **Região Principal:** Holborn, Pimlico (Millbank) e Charing Cross  
* **Clima Previsto:** 4°C a 8°C, ar límpido e frio, perfeito para museus acolhedores.

* **09h30 – 12h00 | O Enigma do Sir John Soane’s Museum (Lincoln's Inn Fields):**  
  Visite a genial casa-museu do arquiteto do século XIX Sir John Soane. Um espetáculo de manipulação da luz zenital, com paredes que se abrem como biombos para revelar gravuras raras de Piranesi e Hogarth, além de um sarcófago egípcio original no subsolo abobadado. É um dos segredos arquitetônicos mais fascinantes da Europa.
* **12h30 – 14h00 | Almoço em Westminster / Millbank:**  
  Almoço no elegante bistrô *Rex Whistler Restaurant* (ou café da Tate Britain) ou um almoço clássico britânico no pub histórico *The Two Chairmen*.
* **14h15 – 17h00 | Tate Britain & Grande Mostra: "Turner and Constable":**  
  Dedique a tarde à **Tate Britain**, o berço da arte britânica. Em fins de novembro de 2025, o museu apresenta a monumental exposição temporária colocando lado a lado **J.M.W. Turner** (o mestre precursor da abstração e da luz atmosférica) e **John Constable** (o mestre do realismo e da paisagem rural). Uma oportunidade histórica para amantes das artes visuais.
* **17h30 – 19h30 | O Bar de Vinhos mais Antigo de Londres: Gordon’s Wine Bar (1890):**  
  Siga até Charing Cross para adentrar o lendário **Gordon’s Wine Bar**. Instalado em caves subterrâneas de pedra, sem luz elétrica direta (apenas velas sobre mesas de barris de carvalho), com poeira histórica nas garrafas de vinho fortificado e uma carta primorosa de Portos, Xerez, Madeiras e tintos europeus acompanhados de queijos maduros.
* **20h00 – 22h00 | Jantar Casual e Retorno:**  
  Caminhada até Covent Garden para um jantar acolhedor no italiano autoral *Bancone* (famoso pelas massas artesanais frescas feitas na hora) ou retorno ao hotel.
* **Transporte:** Metrô (Linha Piccadilly e Victoria Line / Ônibus tradicional nº 87 à beira do rio).
* **Estimativa de Gastos:**
  * *Transporte:* £8,50 (Daily Cap).
  * *Alimentação:* £55 a £85 (almoço + Gordon's Wine Bar + jantar leve).
  * *Atrações:* £22 (ingresso especial da exposição temporária Turner and Constable; acervo permanente gratuito).
  * *Total do Dia:* **~£85 a £115**.

---

### Dia 5: Quinta-feira, 27/11/2025 – Geometria do Design, Parques Reais e Imersão Digital
* **Região Principal:** Kensington, Hyde Park e Marble Arch  
* **Clima Previsto:** 6°C a 11°C, céu parcialmente nublado, brisa outonal nos parques.

* **09h30 – 12h00 | Design Museum & Exposição "Wes Anderson: The Archives":**  
  Em Kensington, visite o premiado **Design Museum**, instalado no icônico edifício modernista reformado por John Pawson com uma espetacular cobertura parabólica hiperbólica de cobre. Explore a imperdível retrospectiva de **Wes Anderson**, analisando cenografia, simetria arquitetônica, paletas de cores e figurinos originais de seus filmes cultuados.
* **12h15 – 13h45 | Caminhada e Almoço em Kensington:**  
  Caminhada charmosa pelo bairro de Kensington High Street e almoço no bistrô francês *Aubaine* ou no tradicional pub vitoriano *The Churchill Arms* (famoso pela fachada monumental coberta de folhagens e luzes).
* **14h00 – 15h30 | Serpentine Galleries & Pavilhões Experimentais (Kensington Gardens):**  
  Caminhe pelos gramados dourados de outono dos Jardins de Kensington até a **Serpentine South & North Gallery**, conhecidas internacionalmente pelos projetos experimentais e pelas intervenções arquitetônicas inovadoras de Zaha Hadid e outros mestres mundiais.
* **16h00 – 18h00 | Tecnologia e Arte Digital no Frameless (Marble Arch):**  
  Dirija-se a Marble Arch para vivenciar a **Frameless**, a maior atração permanente de arte digital imersiva do Reino Unido. Quatro galerias monumentais com projeções mapeadas interativas em 360°, trilha sonora surround e algoritmos de computação visual que desconstroem obras-primas de Cézanne, Kandinsky, Monet, Dalí e Klimt.
* **18h30 – 21h30 | Jantar & Vinhos em Mayfair / Marylebone:**  
  Jantar e taças de vinho no acolhedor bistrô de vinhos **Clarette** (gerenciado por membros da família proprietária do lendário Château Margaux) ou no contemporâneo *28°-50° Wine Workshop*.
* **Transporte:** Metrô (District Line de Southwark/Blackfriars para Kensington; Central Line para Marble Arch).
* **Estimativa de Gastos:**
  * *Transporte:* £8,50 (Daily Cap).
  * *Alimentação:* £60 a £90 (almoço em Kensington + jantar com vinhos em Marylebone).
  * *Atrações:* £47 (£22 ingresso Wes Anderson no Design Museum + £25 ingresso Frameless).
  * *Total do Dia:* **~£115 a £145**.

---

### Dia 6: Sexta-feira, 28/11/2025 – Arte em Telas 8K, Luzes Festivas e a Meca dos Vinhos
* **Região Principal:** Tottenham Court Road, Soho, Covent Garden e Strand  
* **Clima Previsto:** 5°C a 9°C, entardecer vibrante e festivo.

* **09h30 – 11h30 | Outernet London & Now Building (Tottenham Court Road):**  
  Inicie o dia no ápice da tecnologia visual pública: o **Outernet London**. É o maior átrio público coberto do mundo com painéis contínuos de LED em resolução 8K do chão ao teto de quatro andares. Assista a exibições imersivas gratuitas sincronizadas com áudio espacial com temas que misturam inteligência artificial, física quântica e animação hiper-realista.
* **12h00 – 13h30 | Almoço Ágil & Soho Criativo:**  
  Almoço no descolado *Barrafina* (balcão espanhol premiado com vinhos da Catalunha e Rioja) ou no asiático de autor *Kiln*.
* **14h00 – 16h30 | Circuito de Arquitetura e Luzes Urbanas (Carnaby, Regent & Covent Garden):**  
  Com a chegada do entardecer às 16h, caminhe pela **Carnaby Street** para contemplar sua iluminação temática artística e conceitual (famosa por temas não convencionais a cada ano). Desça a curva cenográfica da **Regent Street** admirando os anjos luminosos gigantes (*The Spirit of Christmas*) e finalize na praça de pedestres do **Covent Garden**, com sua árvore monumental e atmosfera de festa.
* **17h00 – 19h00 | Pista de Gelo na Somerset House & Mulled Wine:**  
  Siga para o imponente palácio neoclássico da **Somerset House**. Observe (ou experimente) a famosa pista de patinação no gelo montada no pátio interno com uma árvore de Natal de 12 metros, desfrutando de uma taça aromática de *mulled wine* artesanal no pop-up lounge.
* **19h30 – 22h30 | A Noite do Vinho: Jantar no Noble Rot (Soho ou Bloomsbury):**  
  O ponto alto para amantes de enologia em Londres. Criado pelos fundadores da lendária revista independente sobre vinhos *Noble Rot*, o restaurante oferece uma carta estratosférica com pequenos vignerons europeus, rótulos naturais e raridades servidas pelo sistema Coravin. O menu combina bistronomia britânica e francesa impecável (não perca as ostras com molho de vinagre de vinho tinto e a torta de caça da estação). *Reserva prévia indispensável.*
* **Transporte:** Deslocamentos a pé entre Soho, West End e Covent Garden; metrô no início e fim do dia.
* **Estimativa de Gastos:**
  * *Transporte:* £8,50 (Daily Cap).
  * *Alimentação:* £80 a £130 (almoço de autor no Soho + banquete com taças premium no Noble Rot).
  * *Atrações:* £0 a £15 (Outernet e caminhada luminosa gratuitas; acesso ao pátio da Somerset House gratuito, patinação opcional £15).
  * *Total do Dia:* **~£90 a £150**.

---

### Dia 7: Sábado, 29/11/2025 – Arquitetura Fluvial, Magia de Inverno e Noite de Encerramento
* **Região Principal:** Tâmisa, Westminster e Hyde Park  
* **Clima Previsto:** 4°C a 8°C, clima de festival de inverno com céu limpo ao cair da noite.

* **10h00 – 11h30 | Rota Fluvial Panorâmica no Tâmisa (Uber Boat by Thames Clippers):**  
  Embarque no catamarã expresso no píer de **Bankside**. Desfrute da cidade a partir da água navegando em direção a Westminster: passe sob as pontes históricas, avistando o Tate Modern, o Shakespeare’s Globe, o Parlamento Britânico e a torre do **Big Ben**. Uma perspectiva arquitetônica e urbana incomparável sem o trânsito da superfície.
* **12h00 – 14h00 | Almoço em Mayfair / St. James's:**  
  Desembarque e caminhe pelo bairro aristocrático de St. James’s. Almoço no icônico *The Wolseley* (ambiente grandioso em estilo Art Déco) ou visita à tradicionalíssima loja de vinhos reais **Berry Bros. & Rudd** (fundada em 1698 no número 3 de St James's Street, onde Napoleão III se escondia em suas adegas subterrâneas).
* **14h30 – 18h00 | Hyde Park Winter Wonderland:**  
  Mergulhe na atmosfera festiva do maior parque da cidade. O **Winter Wonderland** reúne chalés de madeira com artesanato, roda-gigante panorâmica com vista de toda Londres, aromas de castanhas assadas e barraquinhas bávaras com sidras e cervejas especiais.
* **19h00 – 22h00 | Jantar de Despedida: Alta Gastronomia Contemporânea:**  
  Celebração final no estrelado **Lyle's** em Shoreditch (menu degustação focado em ingredientes sazonais britânicos e carta de vinhos minimalistas) ou no celebrado **St. John Bread and Wine** (pioneiro no conceito *nose-to-tail* e com uma das seleções de vinhos franceses mais respeitadas do país).
* **Transporte:** Uber Boat (barco pelo Tâmisa) + Metrô / Táxi para a noite de encerramento.
* **Estimativa de Gastos:**
  * *Transporte:* £18 (Uber Boat £9,50 + metrô diário £8,50).
  * *Alimentação:* £80 a £140 (almoço elegante + snacks no festival + jantar especial com harmonização).
  * *Atrações:* £5 a £10 (entrada do Winter Wonderland em horário de pico).
  * *Total do Dia:* **~£105 a £165**.

---

### Dia 8: Domingo, 30/11/2025 – Últimos Aromas, Livrarias de Arte e Retorno ao Brasil
* **Região Principal:** South Bank / Heathrow Airport (LHR)  
* **Clima Previsto:** 5°C a 9°C, manhã outonal tranquila de domingo.

* **09h00 – 11h30 | Manhã Relaxada & Livrarias de Design:**  
  Café da manhã com calma. Faça uma última caminhada pela beira do rio ou explore as excelentes livrarias de arquitetura e design da Tate Modern ou da galeria Hayward no Southbank Centre para garimpar publicações e catálogos de arte contemporânea.
* **12h00 – 14h00 | Check-out e Almoço de Domingo (Sunday Roast Tradicional):**  
  Check-out no hotel (deixando as malas no maleiro, se o voo for no final do dia). Desfrute do ritual dominical britânico por excelência: um autêntico **Sunday Roast** (carne assada com *Yorkshire pudding*, batatas crocantes e molho de vinho) em um pub acolhedor, como o *The George Inn* (o único pub remanescente do século XVII com galerias de madeira onde Charles Dickens costumava beber).
* **15h00 | Deslocamento Estratégico para Heathrow (LHR):**  
  Embarque na **Elizabeth Line** em direção aos Terminais 2, 3 ou 5 do Aeroporto de Heathrow. Trajeto tranquilo, pontual e sem solavancos de cerca de 40 minutos com bagagens.
* **16h30 em diante | Formalidades no Aeroporto e Voo de Retorno a São Paulo:**  
  Check-in, despacho de bagagens, compras de última hora de rótulos livres de impostos no Duty Free de Heathrow e embarque para o voo noturno rumo ao Brasil.
* **Transporte:** Elizabeth Line rumo a Heathrow.
* **Estimativa de Gastos:**
  * *Transporte:* £13,30 (Elizabeth Line até LHR).
  * *Alimentação:* £30 a £50 (Sunday Roast completo + café).
  * *Atrações:* £0.
  * *Total do Dia:* **~£43 a £63**.

---

## 5. Estimativa Consolidada de Gastos

Londres utiliza a Libra Esterlina (GBP / £). Os grandes museus nacionais têm entrada franca para as coleções permanentes, o que permite canalizar o orçamento para grandes mostras temporárias, experiências imersivas e excelência em vinhos e refeições.

### 5.1. Faixas de Orçamento Diário Médio (por pessoa / excluindo voos e hotel)

| Estilo de Viagem | Transporte Diário | Alimentação & Vinhos | Ingressos & Experiências | Média Total / Dia |
| :--- | :--- | :--- | :--- | :--- |
| **Prático / Cultural** | £8,50 (Metrô Cap) | £40 – £55 (street food, bistrôs casuais, 1 taça) | £10 – £20 (1 exibição temporária) | **£60 – £85** |
| **Conforto / Enogastronômico** *(Recomendado)* | £8,50 – £15 (Metrô + 1 barco/táxi) | £70 – £110 (Almoço bom, wine bars autorais, Noble Rot) | £20 – £35 (Mostras temporárias + Frameless) | **£100 – £160** |
| **Premium / Estrelado** | £35 – £60 (Black Cabs sob demanda) | £150 – £250+ (Menus degustação Michelin & garrafas raras) | £40 – £80 (Ingressos VIP e acessos exclusivos) | **£230 – £390+** |

### 5.2. Resumo de Custos dos Principais Pontos e Ingressos do Roteiro
* **Tate Modern, Tate Britain (Coleção Permanente) e Barbican Estate:** **Gratuito (£0)**.
* **Exposição Wes Anderson: The Archives (Design Museum):** **~£22,00**.
* **Exposição Turner & Constable (Tate Britain):** **~£22,00**.
* **Frameless London (Experiência Digital Imersiva):** **~£25,00 a £28,00**.
* **Outernet London (Now Building):** **Gratuito (£0)**.
* **The Garden at 120 (Mirante Panorâmico na City):** **Gratuito (£0)**.
* **Sir John Soane’s Museum:** **Gratuito (£0)** *(requer agendamento prévio online)*.
* **Patinação no Gelo Somerset House:** **~£15,00 a £20,00** *(apenas contemplação do pátio é gratuita)*.
* **Uber Boat by Thames Clippers (Bankside → Westminster):** **~£9,40 a £11,00**.

---

## 6. Mobilidade Urbana, Logística e Transporte

* **Como Pagar o Transporte:** **Não compre bilhetes unitários de papel nem o Oyster Card físico.** Londres utiliza o sistema *Contactless*. Basta aproximar o seu cartão de débito/crédito internacional (como Nomad, Wise, C6 ou cartões de crédito convencionais) ou a carteira digital do smartphone/smartwatch (Apple Pay / Google Wallet) diretamente nas catracas de entrada e saída.
* **O Teto Diário (*Daily Fare Cap*):** Para viagens nas Zonas 1 e 2 (onde estão todas as atrações deste roteiro), a tarifa diária é congelada no valor máximo de **£8,50 por dia**, não importando quantas viagens de metrô você faça. A partir da terceira viagem no mesmo dia, todas as demais saem com custo zero.
* **Conexão com o Aeroporto:** A melhor opção em termos de velocidade, custo e modernidade é a **Elizabeth Line** ligando Heathrow aos eixos centrais de conexão em menos de 45 minutos por cerca de **£13,30**, com trens espaçosos para bagagens, climatizados e acessíveis por elevadores.
* **App Obrigatório de Navegação:** Baixe o **Citymapper**. Ele é infinitely superior ao Google Maps em Londres: indica a melhor saída da estação, o vagão mais conveniente para embarcar, o tempo real de chegada dos ônibus e opções combinadas a pé.

---

## 7. Informações Essenciais de Segurança, Costumes e Dicas de Ouro

### Segurança Prática
1. **Atenção aos Furtos de Smartphone (*Phone Snatching*):** Este é o principal alerta de segurança em Londres nos dias de hoje. Indivíduos em bicicletas elétricas ou patinetes sobem na calçada e arrancam celulares da mão de pessoas que estão distraídas tirando fotos ou checando mapas perto do meio-fio. **Regra de ouro:** nunca use o celular na beirada da calçada voltado para a rua. Posicione-se contra a fachada de um prédio ou entre em uma loja/café para consultar rotas.
2. **Socorro e Emergências:** O telefone nacional de emergência (Polícia, Bombeiros e Ambulância) é o **999** ou **112**. Para reportar incidentes sem perigo iminente (como furto de carteira após o fato ocorrido), disque **101**.
3. **Transporte Noturno:** À noite, utilize exclusivamente os táxis oficiais pretos (*Black Cabs*) sinalizados com luz amarela acesa ou convoque viagens por aplicativos consolidados (**Uber, Bolt, FreeNow**). Jamais aceite ofertas de motoristas informais em portas de casas noturnas ou pubs.

### Costumes Locais & Etiqueta Britânica
* **Escada Rolante no Metrô (*Stand on the right*):** Mantenha-se rigorosamente do **lado direito** dos degraus se estiver parado. O lado esquerdo é exclusivo para as pessoas que estão subindo ou descendo a pé. Bloquear a passagem à esquerda é a gafe número um da cidade.
* **A Sagrada Cultura das Filas (*Queuing*):** Os britânicos respeitam a ordem de chegada de maneira quase sagrada. Em pontos de ônibus, caixas de café ou balcões, observe a fila formada e jamais passe na frente de ninguém.
* **Como Funciona um Pub Inglês:** Na esmagadora maioria dos pubs tradicionais, **não existe garçom na mesa**. Você deve escolher um lugar vago, dirigir-se diretamente ao balcão (*bar*), fazer o seu pedido de cerveja, vinho e comida, realizar o pagamento por aproximação imediatamente e levar as suas próprias bebidas para a mesa (a comida quente é levada à mesa pelos atendentes por meio de um número entregue no balcão).
* **Gorjetas (*Service Charge*):** Nos restaurantes sentados com serviço à mesa, já vem incluída na conta final uma taxa de serviço discricionária de **12,5%**. Se este valor já estiver discriminado na conta, não é necessário nem esperado deixar nada a mais.
* **Cidade Quase 100% Cashless:** Você não precisa se preocupar em trocar grandes somas de dinheiro vivo. Desde banheiros públicos e artistas de rua até barracas do Borough Market e Black Cabs, absolutamente tudo opera por aproximação com cartões digitais.

---

*Com este roteiro estruturado na perfeita convergência de arte de vanguarda, arquitetura de classe mundial, inovação e o melhor da cena enogastronômica londrina, sua semana de 23 a 30 de novembro de 2025 será verdadeiramente inesquecível!*